In [3]:
from pioneer.config.data_configurations import *
from pioneer.config.axes import *

config1 = DataConfiguration(BatchAxes(AxesDim(3), EllipsisDim()), EllipsisAxes(), FeatureAxes(shape=(AxesDim(5),)))
config2 = DataConfiguration(BatchAxes(AxesDim(3)), EllipsisAxes(), FeatureAxes(shape=(AxesDim(5),)))
dict1 = config1.unify_with(config2)[0]
for k, v in dict1.items():
    print(k, v)

(<pioneer.config.axes.BatchAxes object at 0x7a6972cfc8b0>, <pioneer.config.axes.EllipsisAxes object at 0x7a6a445ea260>, <pioneer.config.axes.FeatureAxes object at 0x7a6972d0a4a0>) (<pioneer.config.axes.BatchAxes object at 0x7a6972cfed70>, <pioneer.config.axes.EllipsisAxes object at 0x7a6a445ea1a0>, <pioneer.config.axes.FeatureAxes object at 0x7a6a445ea140>)
FeatureAxes([5]) FeatureAxes([5])
... ...
BatchAxes([3, ...]) BatchAxes([3])
... ...
BatchAxes([3, ...]) {<pioneer.config.axes.AxesDim object at 0x7a6972cfcb20>: <pioneer.config.axes.AxesDim object at 0x7a6a445e9cc0>, <pioneer.config.axes.EllipsisDim object at 0x7a6972cfe470>: ()}
... (<pioneer.config.axes.EllipsisAxes object at 0x7a6a445ea1a0>,)
FeatureAxes([5]) {<pioneer.config.axes.AxesDim object at 0x7a6972d0bbb0>: <pioneer.config.axes.AxesDim object at 0x7a6a445ea5c0>}


In [2]:
from pioneer import *
from typing import Annotated, Any
import torch

class ReIm(BackendNode):
    def forward(
        self, x) -> tuple[Annotated[Any, DataConfiguration()],
                          Annotated[Any, DataConfiguration()]]:
        return self.implementation(x)

    def torch_implementation(self, x):
        return x.real, x.imag

    def tensorflow_implementation(self, x):
        return self.backend.library.math.real(x), self.backend.library.math.imag(x)

class Re(TrackedNode):
    def __init__(
        self, backend = DEFAULT_DL_BACKEND, name="complex_valued"
        ):
        self.re = ReIm(backend=backend)
        super().__init__(name=name)

    def forward(self, x):
        re, im = self.re(x)

        return re + im

a = Re()
t = torch.zeros(3,4, dtype=torch.complex64)
a(t)

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])